# Session 2. Transformer

## 목표
**GPT 논문을 읽을 수 있다.**

이번 세션의 목표는 Transformer 구조를 “외워서 설명”하는 것이 아니라,  
GPT 계열 논문을 읽을 때 반복적으로 등장하는 핵심 개념을 이해하는 것입니다.

다루는 흐름은 다음과 같습니다.

1. 왜 RNN 기반 구조가 한계에 부딪혔는가
2. Attention은 무엇을 해결했는가
3. Query / Key / Value는 어떤 의미인가
4. Multi-Head Attention은 왜 필요한가
5. Positional Encoding은 왜 필요한가
6. Transformer Block은 어떻게 구성되는가
7. KV Cache는 왜 LLM inference의 핵심인가
8. 대표 논문: *Attention Is All You Need*, *RoFormer*

---

# 1. Why RNN Failed

Transformer가 등장하기 전, 자연어 처리는 RNN, LSTM, GRU 같은 순차 모델이 중심이었습니다.

RNN은 문장을 왼쪽에서 오른쪽으로 하나씩 읽으면서 hidden state를 업데이트합니다.

```text
x1 → x2 → x3 → x4 → ...
```

이 방식은 직관적이지만 두 가지 큰 문제가 있습니다.

## 1.1 Long Dependency

문장이 길어질수록 앞쪽 정보가 뒤쪽까지 잘 전달되지 않습니다.

예를 들어:

> The patient who had been admitted to the ICU after severe pneumonia and multiple complications **was discharged**.

`patient`와 `was discharged`는 의미적으로 연결되어 있지만, 두 단어 사이에는 많은 정보가 끼어 있습니다.

RNN은 이런 장거리 의존성을 hidden state 하나에 압축해서 전달해야 합니다.

결과적으로 긴 문장에서는 중요한 정보가 희석되기 쉽습니다.

## 1.2 Sequential Bottleneck

RNN은 이전 시점의 계산이 끝나야 다음 시점을 계산할 수 있습니다.

```text
h1 계산 → h2 계산 → h3 계산 → h4 계산
```

즉, 병렬화가 어렵습니다.

GPU는 병렬 계산에 강한데, RNN 구조는 이 장점을 충분히 활용하지 못합니다.

Transformer는 이 문제를 Attention으로 해결합니다.

---

# 2. Attention

Attention의 핵심 아이디어는 간단합니다.

> 현재 단어를 이해할 때, 문장 안의 다른 단어들을 직접 참고하자.

RNN은 정보를 순차적으로 전달합니다.  
Attention은 모든 토큰이 서로를 직접 바라볼 수 있게 합니다.

```text
각 token → 모든 token을 참고
```

예를 들어 문장에서 `discharged`라는 단어를 이해할 때,  
모델은 앞에 있는 `patient`, `ICU`, `pneumonia` 같은 단어에 직접 attention을 줄 수 있습니다.

Attention은 다음 질문에 답하는 구조입니다.

> 지금 이 token은 문장 안의 어떤 token을 얼마나 참고해야 하는가?

---

# 3. Query, Key, Value

Attention은 Query, Key, Value로 구성됩니다.

각 token은 세 가지 벡터로 변환됩니다.

| 구성요소 | 직관적 의미 |
|---|---|
| Query | 내가 찾고 싶은 정보 |
| Key | 내가 가진 정보의 이름표 |
| Value | 실제 전달할 정보 |

비유하면 다음과 같습니다.

- Query: “나는 어떤 정보를 찾고 있나?”
- Key: “나는 어떤 종류의 정보인가?”
- Value: “내가 실제로 줄 정보는 무엇인가?”

Attention score는 Query와 Key의 유사도로 계산됩니다.

Query와 Key가 잘 맞으면 해당 token의 Value를 많이 가져옵니다.

## 3.1 Scaled Dot-Product Attention

Transformer에서 사용하는 attention은 다음과 같습니다.

\[
Attention(Q, K, V) = softmax \left( \frac{QK^T}{\sqrt{d_k}} \right) V
\]

의미는 다음 순서입니다.

1. Query와 Key의 유사도를 계산한다.
2. 너무 값이 커지지 않도록 \(\sqrt{d_k}\)로 나눈다.
3. softmax로 attention weight를 만든다.
4. attention weight를 Value에 곱해 필요한 정보를 섞는다.

In [ ]:
# Scaled Dot-Product Attention을 직접 구현해봅니다.

import torch
import torch.nn.functional as F

torch.manual_seed(42)

# 예시: token 4개, embedding dimension 8
seq_len = 4
d_model = 8
d_k = 8

X = torch.randn(seq_len, d_model)

# 실제 Transformer에서는 W_Q, W_K, W_V가 학습됩니다.
W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

scores = Q @ K.T / (d_k ** 0.5)
attention_weights = F.softmax(scores, dim=-1)
output = attention_weights @ V

print("Input shape:", X.shape)
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
print("Attention weights shape:", attention_weights.shape)
print("Output shape:", output.shape)

print("\nAttention weights:")
print(attention_weights)

위 결과에서 `attention_weights`는 각 token이 다른 token을 얼마나 참고하는지 보여줍니다.

shape이 `(4, 4)`인 이유는 token 4개가 각각 token 4개를 바라보기 때문입니다.

```text
token 1 → token 1,2,3,4
token 2 → token 1,2,3,4
token 3 → token 1,2,3,4
token 4 → token 1,2,3,4
```

---

# 4. Multi-Head Attention

하나의 attention만 쓰면 한 가지 관점으로만 token 관계를 봅니다.

하지만 문장에는 여러 종류의 관계가 있습니다.

- 문법적 관계
- 의미적 관계
- 주어-동사 관계
- 앞뒤 문맥 관계
- 지시어 관계

Multi-Head Attention은 attention을 여러 개의 head로 나누어,  
각 head가 서로 다른 관계를 학습하게 합니다.

```text
Head 1: 문법 관계
Head 2: 의미 관계
Head 3: 지시어 관계
Head 4: 위치 관계
```

실제로 각 head가 반드시 사람이 해석 가능한 역할로 나뉘는 것은 아니지만,  
여러 표현 공간에서 attention을 수행한다는 것이 핵심입니다.

In [ ]:
# PyTorch의 MultiheadAttention을 사용해봅니다.

import torch
import torch.nn as nn

torch.manual_seed(42)

seq_len = 5
batch_size = 2
d_model = 16
num_heads = 4

x = torch.randn(seq_len, batch_size, d_model)

mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=False)

attn_output, attn_weights = mha(x, x, x)

print("Input shape:", x.shape)
print("Output shape:", attn_output.shape)
print("Attention weights shape:", attn_weights.shape)

`x, x, x`를 넣은 이유는 Self-Attention이기 때문입니다.

Self-Attention에서는 같은 입력에서 Query, Key, Value를 모두 만듭니다.

```python
mha(x, x, x)
```

즉, 문장 안의 token들이 자기들끼리 서로를 참고합니다.

---

# 5. Positional Encoding

Attention은 모든 token을 한 번에 비교합니다.

이 구조는 병렬화에는 좋지만, 한 가지 문제가 있습니다.

> Attention 자체는 token의 순서를 모릅니다.

예를 들어 다음 두 문장은 단어는 같지만 의미가 다릅니다.

```text
dog bites man
man bites dog
```

Attention만 쓰면 token 집합은 같기 때문에 순서 정보를 따로 넣어줘야 합니다.

이때 사용하는 것이 Positional Encoding입니다.

## 5.1 Sinusoidal Positional Encoding

*Attention Is All You Need* 논문에서는 sin, cos 함수를 사용한 positional encoding을 제안했습니다.

\[
PE(pos, 2i) = \sin \left( \frac{pos}{10000^{2i/d_{model}}} \right)
\]

\[
PE(pos, 2i+1) = \cos \left( \frac{pos}{10000^{2i/d_{model}}} \right)
\]

특징은 다음과 같습니다.

- 위치마다 고유한 벡터를 만든다.
- 학습 파라미터 없이 계산 가능하다.
- 상대적 위치 관계를 표현하는 데 유리하다.

In [ ]:
# Sinusoidal Positional Encoding 시각화

import math
import torch
import matplotlib.pyplot as plt

def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

max_len = 50
d_model = 16

pe = sinusoidal_positional_encoding(max_len, d_model)

plt.figure(figsize=(10, 4))
plt.imshow(pe.T, aspect="auto")
plt.xlabel("Position")
plt.ylabel("Dimension")
plt.title("Sinusoidal Positional Encoding")
plt.colorbar()
plt.show()

위 그림에서 x축은 token 위치, y축은 embedding dimension입니다.

각 위치는 서로 다른 패턴의 벡터를 갖습니다.  
Transformer는 token embedding에 positional encoding을 더해서 순서 정보를 함께 사용합니다.

## 5.2 RoPE: Rotary Positional Embedding

RoPE는 *RoFormer* 논문에서 제안된 방식입니다.

Sinusoidal encoding처럼 위치 벡터를 더하는 것이 아니라,  
Query와 Key 벡터를 위치에 따라 회전시킵니다.

핵심 아이디어는 다음과 같습니다.

> 위치 정보를 embedding에 더하지 말고, attention 계산 자체에 상대 위치 정보가 반영되도록 하자.

RoPE는 LLaMA 계열 등 현대 LLM에서 널리 사용되는 positional encoding 방식입니다.

직관적으로 보면, 각 token의 Query/Key 벡터가 위치에 따라 조금씩 회전하고,  
두 token 사이의 상대적 위치가 attention score에 자연스럽게 반영됩니다.

In [ ]:
# RoPE의 핵심 직관: 2차원 벡터를 위치에 따라 회전시키기

import torch
import math
import matplotlib.pyplot as plt

def rotate_vector(v, theta):
    rotation = torch.tensor([
        [math.cos(theta), -math.sin(theta)],
        [math.sin(theta),  math.cos(theta)]
    ], dtype=torch.float32)
    return rotation @ v

v = torch.tensor([1.0, 0.0])

positions = list(range(8))
rotated = torch.stack([rotate_vector(v, pos * 0.4) for pos in positions])

plt.figure(figsize=(5, 5))
for i, vec in enumerate(rotated):
    plt.arrow(0, 0, vec[0].item(), vec[1].item(), head_width=0.03, length_includes_head=True)
    plt.text(vec[0].item(), vec[1].item(), str(i))

plt.xlim(-1.2, 1.2)
plt.ylim(-1.2, 1.2)
plt.axhline(0, linewidth=0.5)
plt.axvline(0, linewidth=0.5)
plt.title("RoPE intuition: position-dependent rotation")
plt.grid(True)
plt.show()

---

# 6. Transformer Block

Transformer는 attention 하나만으로 구성되지 않습니다.

기본 block은 다음 요소들로 이루어집니다.

```text
Input
  ↓
Self-Attention
  ↓
Residual Connection
  ↓
LayerNorm
  ↓
Feed Forward Network
  ↓
Residual Connection
  ↓
LayerNorm
  ↓
Output
```

GPT 계열 모델은 decoder-only Transformer입니다.

즉, 다음 token을 예측하기 위해 이전 token들만 볼 수 있도록 causal mask를 사용합니다.

## 6.1 Attention

Attention은 token 간 관계를 학습합니다.

예를 들어 문장 안에서 어떤 단어가 어떤 단어를 참고해야 하는지를 계산합니다.

## 6.2 FFN

FFN은 각 token에 독립적으로 적용되는 작은 MLP입니다.

Attention이 token 사이의 관계를 섞는 역할이라면,  
FFN은 각 token의 표현을 더 복잡하게 변환하는 역할입니다.

## 6.3 Residual Connection

Residual connection은 입력을 출력에 더합니다.

\[
y = x + F(x)
\]

이 구조는 깊은 모델의 학습을 안정화합니다.

## 6.4 LayerNorm

LayerNorm은 각 token의 hidden dimension을 정규화합니다.

큰 모델에서는 activation scale이 불안정해지기 쉬운데,  
LayerNorm은 학습을 안정화하는 역할을 합니다.

In [ ]:
# 아주 작은 Transformer Block 구현

import torch
import torch.nn as nn

class TinyTransformerBlock(nn.Module):
    def __init__(self, d_model=32, num_heads=4, d_ff=128):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, attn_mask=None):
        attn_out, _ = self.attn(x, x, x, attn_mask=attn_mask)
        x = self.ln1(x + attn_out)

        ffn_out = self.ffn(x)
        x = self.ln2(x + ffn_out)
        return x

batch_size = 2
seq_len = 6
d_model = 32

x = torch.randn(batch_size, seq_len, d_model)

block = TinyTransformerBlock(d_model=d_model, num_heads=4)
out = block(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

---

# 7. Causal Mask

GPT는 다음 token을 예측하는 모델입니다.

따라서 현재 token이 미래 token을 보면 안 됩니다.

예를 들어:

```text
I love deep learning
```

`love`를 처리할 때 `deep`, `learning`을 미리 보면 정답을 훔쳐보는 것이 됩니다.

그래서 GPT는 causal mask를 사용합니다.

```text
token 1 → token 1만 볼 수 있음
token 2 → token 1,2만 볼 수 있음
token 3 → token 1,2,3만 볼 수 있음
```

In [ ]:
# Causal mask 만들기

import torch
import matplotlib.pyplot as plt

seq_len = 8

# True인 위치는 attention을 막을 위치입니다.
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

print(causal_mask)

plt.figure(figsize=(5, 5))
plt.imshow(causal_mask)
plt.title("Causal Mask")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.show()

---

# 8. KV Cache

LLM inference에서 KV Cache는 매우 중요합니다.

Transformer는 새 token을 생성할 때마다 attention을 계산합니다.

문제는 autoregressive generation에서는 token을 하나씩 생성한다는 점입니다.

```text
입력 prompt → 다음 token 생성 → 그 다음 token 생성 → ...
```

매번 전체 prompt에 대해 Key, Value를 다시 계산하면 매우 비효율적입니다.

KV Cache는 이전 token들의 Key, Value를 저장해두고 재사용하는 기법입니다.

## 8.1 왜 필요한가

예를 들어 prompt 길이가 1,000 token이고, 새 token을 100개 생성한다고 합시다.

KV Cache가 없다면 매 step마다 기존 1,000개 이상의 token에 대해 Key/Value를 다시 계산해야 합니다.

KV Cache가 있으면 이전 token의 Key/Value는 저장해두고,  
새로 들어온 token의 Key/Value만 계산하면 됩니다.

이 덕분에 decode 단계가 훨씬 빨라집니다.

## 8.2 Prefill과 Decode

LLM inference는 보통 두 단계로 나눕니다.

## Prefill

사용자가 입력한 prompt 전체를 한 번에 처리합니다.

```text
prompt tokens 전체 입력
→ 각 layer의 K/V cache 생성
```

Prefill은 병렬화가 잘 됩니다.  
하지만 prompt가 길수록 메모리와 계산량이 커집니다.

## Decode

이후 token을 하나씩 생성합니다.

```text
새 token 1개 입력
→ cache된 K/V와 attention
→ 다음 token 생성
```

Decode는 token을 하나씩 생성하므로 순차적입니다.  
LLM serving에서 latency가 중요한 이유가 여기에 있습니다.

In [ ]:
# KV Cache의 shape 직관

batch_size = 1
num_heads = 4
seq_len = 10
head_dim = 8

# 한 layer에서 저장되는 Key/Value cache
key_cache = torch.randn(batch_size, num_heads, seq_len, head_dim)
value_cache = torch.randn(batch_size, num_heads, seq_len, head_dim)

print("Key cache shape:", key_cache.shape)
print("Value cache shape:", value_cache.shape)

# 새 token이 하나 생성되면 seq_len 방향으로 cache가 늘어납니다.
new_key = torch.randn(batch_size, num_heads, 1, head_dim)
new_value = torch.randn(batch_size, num_heads, 1, head_dim)

key_cache = torch.cat([key_cache, new_key], dim=2)
value_cache = torch.cat([value_cache, new_value], dim=2)

print("Updated key cache shape:", key_cache.shape)
print("Updated value cache shape:", value_cache.shape)

---

# 9. GPT 논문을 읽을 때 보는 포인트

GPT 계열 논문을 읽을 때는 다음 질문을 가지고 보면 좋습니다.

## 9.1 모델 구조

- Decoder-only Transformer인가?
- Attention 구조는 어떤 변형을 쓰는가?
- Positional Encoding은 무엇을 쓰는가?
- LayerNorm 위치는 Pre-LN인가, Post-LN인가?
- FFN activation은 무엇인가?
- context length는 얼마인가?

## 9.2 학습 방식

- next token prediction인가?
- instruction tuning을 했는가?
- RLHF 또는 preference optimization을 썼는가?
- 데이터 규모와 품질은 어떻게 설명되는가?

## 9.3 Inference 최적화

- KV Cache를 사용하는가?
- 긴 context를 어떻게 처리하는가?
- serving latency나 throughput을 어떻게 개선하는가?

---

# 10. 대표 논문

## 10.1 Attention Is All You Need

Transformer의 원형을 제안한 논문입니다.

핵심 기여:

- RNN/CNN 없이 attention만으로 sequence modeling 수행
- Multi-Head Attention 제안
- Sinusoidal Positional Encoding 사용
- Encoder-Decoder Transformer 구조 제안

GPT를 이해하려면 이 논문의 attention, positional encoding, residual connection, layer normalization 개념을 먼저 이해해야 합니다.

## 10.2 RoFormer

RoPE, 즉 Rotary Positional Embedding을 제안한 논문입니다.

핵심 기여:

- Query/Key 벡터를 위치에 따라 회전
- 상대 위치 정보가 attention score에 자연스럽게 반영
- 긴 context 처리에 유리한 positional encoding 방향 제시

현대 LLM에서 RoPE는 매우 자주 등장합니다.

---

# 11. 오늘의 핵심 요약

1. RNN은 long dependency와 sequential bottleneck 때문에 대규모 병렬 학습에 불리했습니다.
2. Attention은 모든 token이 서로를 직접 참고하게 만듭니다.
3. Query는 찾는 정보, Key는 매칭 기준, Value는 실제 전달 정보입니다.
4. Multi-Head Attention은 여러 관점에서 token 관계를 학습합니다.
5. Positional Encoding은 attention에 순서 정보를 제공합니다.
6. Transformer Block은 Attention, FFN, Residual, LayerNorm으로 구성됩니다.
7. GPT는 미래 token을 보지 않기 위해 causal mask를 사용합니다.
8. KV Cache는 LLM inference에서 decode 속도를 높이는 핵심 기술입니다.
9. Prefill은 prompt 전체 처리, Decode는 token-by-token 생성입니다.
10. GPT 논문을 읽을 때는 구조, 학습 방식, inference 최적화를 구분해서 보면 됩니다.

---

# 12. 실습 과제

## 과제 1
Scaled Dot-Product Attention 코드에서 `seq_len`을 4, 8, 16으로 바꿔보고 attention matrix의 shape이 어떻게 변하는지 확인하세요.

## 과제 2
Causal mask를 적용하지 않은 attention과 적용한 attention의 차이를 설명해보세요.

## 과제 3
Sinusoidal Positional Encoding에서 `d_model`을 16, 64, 128로 바꿔보고 시각화 패턴이 어떻게 달라지는지 확인하세요.

## 과제 4
KV Cache가 없을 때와 있을 때 decode 단계에서 계산량이 어떻게 달라지는지 말로 설명해보세요.

## 과제 5
*Attention Is All You Need* 논문의 Abstract와 Introduction을 읽고, RNN을 사용하지 않는다는 점이 왜 중요한지 정리해보세요.